# Real-World Data Project: Retail Sales Analysis

## Objective
Analyze a real-world online retail transaction dataset to understand sales trends, top products, customer activity, and country-wise sales.

## Dataset
**Online Retail** from the UCI Machine Learning Repository.

The dataset contains transactions from a UK-based non-store online retailer between December 2010 and December 2011.

## Tools Used
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- UCI Machine Learning Repository (`ucimlrepo`)

## Main Tasks
1. Load the real-world dataset
2. Inspect and clean the data
3. Create a Sales/Revenue column
4. Analyze monthly sales
5. Find top-selling products
6. Analyze country-wise sales
7. Analyze customer spending
8. Visualize important findings
9. Write conclusions


In [ ]:
# Install the required package if needed
# Run this cell only once if ucimlrepo is not installed

!pip install ucimlrepo pandas numpy matplotlib seaborn openpyxl


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

print("Libraries imported successfully!")


## 1. Load the Real-World Dataset

In [ ]:
# Fetch the Online Retail dataset from UCI
online_retail = fetch_ucirepo(id=352)

df = online_retail.data.features.copy()

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))


In [ ]:
# Display the first 5 rows
df.head()


In [ ]:
# Display column names
print("Columns:")
print(df.columns.tolist())


In [ ]:
# Dataset information
df.info()


In [ ]:
# Basic statistics
df.describe(include="all").T


## 2. Check Missing Values

In [ ]:
missing = df.isnull().sum()

print("Missing values:")
print(missing)


In [ ]:
# Show percentage of missing values
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_table = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": missing_percent.round(2)
})

missing_table


## 3. Data Cleaning

In [ ]:
# Remove rows without a product description
df = df.dropna(subset=["Description"])

# CustomerID is not needed for basic sales analysis
# Keep rows without CustomerID because they still contain valid sales information

# Convert date column
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Remove rows with zero or negative quantity
df = df[df["Quantity"] > 0]

# Remove rows with zero or negative price
df = df[df["UnitPrice"] > 0]

print("Rows after cleaning:", len(df))


In [ ]:
# Remove duplicate records
df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))


## 4. Create Sales Column

In [ ]:
# Revenue for each transaction line
df["Sales"] = df["Quantity"] * df["UnitPrice"]

print("Sales column created.")
df[["Description", "Quantity", "UnitPrice", "Sales"]].head()


In [ ]:
print("Total Sales: £", round(df["Sales"].sum(), 2))
print("Average Transaction Line Sales: £", round(df["Sales"].mean(), 2))


## 5. Monthly Sales Analysis

In [ ]:
# Create month column
df["Month"] = df["InvoiceDate"].dt.to_period("M").astype(str)

monthly_sales = df.groupby("Month")["Sales"].sum().reset_index()

monthly_sales


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_sales["Month"],
    monthly_sales["Sales"],
    marker="o"
)

plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales (£)")
plt.xticks(rotation=45)
plt.grid(True)

plt.show()


In [ ]:
# Month with highest sales
best_month = monthly_sales.loc[
    monthly_sales["Sales"].idxmax()
]

print("Highest Sales Month:", best_month["Month"])
print("Sales: £", round(best_month["Sales"], 2))


## 6. Top 10 Products by Sales

In [ ]:
top_products = (
    df.groupby("Description")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

top_products


In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_products,
    x="Sales",
    y="Description"
)

plt.title("Top 10 Products by Sales")
plt.xlabel("Sales (£)")
plt.ylabel("Product")

plt.show()


## 7. Top 10 Products by Quantity Sold

In [ ]:
top_quantity = (
    df.groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

top_quantity


In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_quantity,
    x="Quantity",
    y="Description"
)

plt.title("Top 10 Products by Quantity Sold")
plt.xlabel("Quantity")
plt.ylabel("Product")

plt.show()


## 8. Country-wise Sales Analysis

In [ ]:
country_sales = (
    df.groupby("Country")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

country_sales


In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=country_sales,
    x="Sales",
    y="Country"
)

plt.title("Top 10 Countries by Sales")
plt.xlabel("Sales (£)")
plt.ylabel("Country")

plt.show()


## 9. Customer Analysis

In [ ]:
# Customer sales
customer_sales = (
    df.dropna(subset=["CustomerID"])
    .groupby("CustomerID")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

customer_sales


In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=customer_sales,
    x="Sales",
    y="CustomerID"
)

plt.title("Top 10 Customers by Sales")
plt.xlabel("Sales (£)")
plt.ylabel("Customer ID")

plt.show()


## 10. Sales by Day of the Week

In [ ]:
df["Day"] = df["InvoiceDate"].dt.day_name()

day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

day_sales = (
    df.groupby("Day")["Sales"]
    .sum()
    .reindex(day_order)
)

day_sales


In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    x=day_sales.index,
    y=day_sales.values
)

plt.title("Sales by Day of the Week")
plt.xlabel("Day")
plt.ylabel("Sales (£)")
plt.xticks(rotation=30)

plt.show()


## 11. Correlation Analysis

In [ ]:
numeric_data = df[["Quantity", "UnitPrice", "Sales"]]

plt.figure(figsize=(7, 5))

sns.heatmap(
    numeric_data.corr(),
    annot=True,
    fmt=".2f"
)

plt.title("Correlation Between Numerical Variables")

plt.show()


## 12. Final Findings

In [ ]:
total_sales = df["Sales"].sum()
best_month_name = monthly_sales.loc[monthly_sales["Sales"].idxmax(), "Month"]
best_product = top_products.iloc[0]["Description"]
best_country = country_sales.iloc[0]["Country"]

print("=" * 55)
print("FINAL RETAIL DATA ANALYSIS")
print("=" * 55)

print("Total Sales: £", round(total_sales, 2))
print("Highest Sales Month:", best_month_name)
print("Top Product by Sales:", best_product)
print("Top Country by Sales:", best_country)

print("=" * 55)


## 13. Conclusion

The analysis provides an end-to-end view of a real-world retail transaction dataset.

### Key conclusions
- Monthly sales show how revenue changes over time.
- A small number of products contribute a large share of sales.
- Country-wise analysis shows where most revenue is generated.
- Customer analysis identifies high-value customers.
- Day-wise analysis helps understand purchasing patterns.
- Visualizations make the sales trends and business patterns easier to understand.

### Expected Outcome
This project demonstrates how Python and data science techniques can be applied to a real-world retail business problem for data cleaning, analysis, visualization, and business insight generation.
